### Implementation of Viterbi Algorithm for Sequence Analysis and Log Probability Calculation

Initialize model parameters: state space, transition matrix, and emission probabilities.

In [1]:
import math

stateSet = ['E', '5', 'I']

transitionMatrix = {
    'E': {'E': math.log(0.9), '5': math.log(0.1), 'I':float('-inf')},
    '5': {'E':float('-inf'), '5':float('-inf'), 'I': math.log(1.0)},
    'I': {'E':float('-inf'), '5':float('-inf'), 'I': math.log(0.9)}
}

emissionMatrix = {
    'E': {'A': math.log(0.25), 'C': math.log(0.25), 'G': math.log(0.25), 'T': math.log(0.25)},
    '5': {'A': math.log(0.05), 'C': float('-inf'), 'G': math.log(0.95), 'T': float('-inf')},
    'I': {'A': math.log(0.4), 'C': math.log(0.1), 'G': math.log(0.1), 'T': math.log(0.4)}
}


Viterbi algorithm execution for finding the most probable state sequence.

In [2]:
dnaSequence = "AGGCTCTCCATAAGG"
sequenceLength = len(dnaSequence)
viterbiTable = [{} for _ in range(sequenceLength)]
pointerTable = [{} for _ in range(sequenceLength)]

# Initialize the first column of Viterbi table
for currentState in stateSet:
    if(currentState == 'E'):
        viterbiTable[0][currentState] = emissionMatrix[currentState][dnaSequence[0]]
    else:
        viterbiTable[0][currentState] = float('-inf')
        
# Dynamic programming to fill Viterbi and backtracking tables
for position in range(1, sequenceLength):
    for currentState in stateSet:
        maxProbability = float('-inf')
        bestPreviousState = ""
        for previousState in stateSet:
            probability = viterbiTable[position-1][previousState] + transitionMatrix[previousState][currentState] + emissionMatrix[currentState][dnaSequence[position]]
            if probability > maxProbability:
                maxProbability = probability
                bestPreviousState = previousState
        viterbiTable[position][currentState] = maxProbability
        pointerTable[position][currentState] = bestPreviousState

# Trace back to find optimal state path
optimalProbability = float('-inf')
optimalFinalState = ''
for currentState in stateSet:
    if viterbiTable[sequenceLength-1][currentState] > optimalProbability:
        optimalProbability = viterbiTable[sequenceLength-1][currentState]
        optimalFinalState = currentState
        
optimalPath = [optimalFinalState]
for timeStep in range(sequenceLength-1, 0, -1):
    optimalFinalState = pointerTable[timeStep][optimalFinalState]
    optimalPath.insert(0, optimalFinalState)
    
print(''.join([state for state in optimalPath]))

EEEEEEEEEEEEEEE


Computing log probability for a specified DNA sequence given its state path annotation.

In [4]:
testSequence = "CTTCATGTGAAAGCAGACGTAAGTCA"
statePath = "EEEEEEEEEEEEEEEEEE5IIIIIII"

def calculateLogProbability(sequence, statePath):
    """Calculate the log probability of a sequence given a state path"""
    assert len(statePath) == len(sequence), "State path and DNA sequence must have equal lengths"

    sequenceLength = len(sequence)

    totalLogProb = 0.0
    for position in range(sequenceLength):
        if position == 0:
            if(statePath[position] != 'E'):
                totalLogProb = float('-inf')
                break
            else:
                totalLogProb = emissionMatrix[statePath[position]][sequence[position]]
        else:
            totalLogProb += transitionMatrix[statePath[position-1]][statePath[position]] + emissionMatrix[statePath[position]][sequence[position]]

    # Add transition to end state (terminal probability)
    totalLogProb += math.log(0.1)
    
    return totalLogProb

print(f"Log probability of sequence given state path: {calculateLogProbability(testSequence, statePath):.2f}")

Log probability of sequence given state path: -41.22
